# 01 Boundaries and the service crosswalk

Fetches fire and rescue authority polygons from the Office for National Statistics
feature service, reconciles them against the territories recorded in the incident
panel, and writes `config/service_crosswalk.csv`.

Run after `scripts/convert_raw.py`.

In [1]:
import pandas as pd

from fdh import boundaries as bd
from fdh import contract
from fdh.config import load_config, repo_root

config = load_config()
raw_dir = config.paths.resolved("raw_dir")
interim_dir = config.paths.resolved("interim_dir")
raw_dir, interim_dir

(WindowsPath('D:/Chris Masters/Final Research Project/Chris-MScfire-demand-horizon/fire-demand-horizon/data/raw'),
 WindowsPath('D:/Chris Masters/Final Research Project/Chris-MScfire-demand-horizon/fire-demand-horizon/data/interim'))

## Fetch

The layer is served in EPSG:27700, which matches the HadUK-Grid projection, so no
reprojection is needed before the aggregation.

In [2]:
layer = config.sources["ons_fra_boundaries"][0]
collection = bd.fetch_layer(layer.url)
record = bd.save_layer(collection, raw_dir / "boundaries" / layer.filename)
record

{'filename': 'fra_2022_en_bgc.geojson',
 'features': 44,
 'properties': ['BNG_E',
  'BNG_N',
  'FID',
  'FRA22CD',
  'FRA22NM',
  'GlobalID',
  'LAT',
  'LONG',
  'Shape__Area',
  'Shape__Length'],
 'crs': {'type': 'name', 'properties': {'name': 'EPSG:27700'}},
 'sha256': '78faea92daebd13ef80f043a020de75a589c0721509f36e8925fc0f736e7f934',
 'bytes': 5092876}

## Reconcile

The join is on E code first and normalised name second.

In [3]:
polygons, code_field, name_field = bd.load_layer(raw_dir / "boundaries" / layer.filename)
territories, by_year = bd.territories_from_panel(interim_dir)

print(f"{len(polygons)} boundary features ({code_field}, {name_field})")
print(f"{len(territories)} incident territories")

44 boundary features (FRA22CD, FRA22NM)
43 incident territories


In [4]:
rows = bd.build_crosswalk(territories, polygons)
crosswalk = pd.DataFrame(rows)
crosswalk

,e_code_territory,frs_territory,boundary_code,boundary_name,match_basis,name_differs
0,E31000001,Avon,E31000001,Avon,e_code,False
1,E31000002,Bedfordshire,E31000002,Bedfordshire,e_code,False
2,E31000003,Berkshire,E31000003,Royal Berkshire,e_code,True
3,E31000004,Buckinghamshire,E31000004,Buckinghamshire & Milton Keynes,e_code,True
4,E31000005,Cambridgeshire,E31000005,Cambridgeshire,e_code,False
5,E31000006,Cheshire,E31000006,Cheshire,e_code,False
6,E31000007,Cleveland,E31000007,Cleveland,e_code,False
7,E31000008,Cornwall and Isles of Scilly,E31000008,Cornwall,e_code,True
8,E31000009,Cumbria,E31000009,Cumbria,e_code,False
9,E31000010,Derbyshire,E31000010,Derbyshire,e_code,False


## What did not line up

In [5]:
print("unmatched incident territories:")
print(crosswalk.loc[crosswalk.match_basis == "unmatched"].to_string(index=False) or "  none")

print("\nmatched on name, E codes differ:")
print(crosswalk.loc[crosswalk.match_basis == "normalised_name"].to_string(index=False))

print("\nboundary features with no incident territory:")
for code, name in bd.unmatched_boundaries(rows, polygons):
    print(f"  {code}  {name}")

unmatched incident territories:
Empty DataFrame
Columns: [e_code_territory, frs_territory, boundary_code, boundary_name, match_basis, name_differs]
Index: []

matched on name, E codes differ:
e_code_territory               frs_territory boundary_code               boundary_name     match_basis  name_differs
       E31000017 Hampshire and Isle of Wight     E31000048 Hampshire and Isle of Wight normalised_name         False

boundary features with no incident territory:
  E31000039  Isles of Scilly


In [6]:
crosswalk.loc[crosswalk.name_differs, ["e_code_territory", "frs_territory", "boundary_name"]]

,e_code_territory,frs_territory,boundary_name
2,E31000003,Berkshire,Royal Berkshire
3,E31000004,Buckinghamshire,Buckinghamshire & Milton Keynes
7,E31000008,Cornwall and Isles of Scilly,Cornwall
12,E31000013,Durham,County Durham and Darlington
16,E31000046,Greater London,London Fire and Emergency Planning Authority
31,E31000030,Nottinghamshire,Nottinghamshire and City of Nottingham
35,E31000033,Staffordshire,Stoke-on-Trent and Staffordshire


## Coverage by financial year

Authorities entering or leaving the series show here.

In [7]:
counts = {year: len(codes) for year, codes in sorted(by_year.items())}
everything = set().union(*by_year.values())
missing = {
    year: sorted((code, territories[code]) for code in everything - codes)
    for year, codes in sorted(by_year.items())
    if everything - codes
}
counts, missing

({'2010/11': 43,
  '2011/12': 43,
  '2012/13': 43,
  '2013/14': 43,
  '2014/15': 43,
  '2015/16': 43,
  '2016/17': 43,
  '2017/18': 43,
  '2018/19': 43,
  '2019/20': 43,
  '2020/21': 43,
  '2021/22': 43,
  '2022/23': 43,
  '2023/24': 43,
  '2024/25': 43,
  '2025/26': 43},
 {})

## Write the crosswalk

The `note` column is for decisions taken by hand: which polygon a merged
territory uses, and what happens to any boundary feature with no territory.
Notes already in the file are carried over when it is rewritten.

In [8]:
out = repo_root() / "config" / "service_crosswalk.csv"
bd.write_crosswalk(rows, out)
print(f"wrote {out}")

wrote D:\Chris Masters\Final Research Project\Chris-MScfire-demand-horizon\fire-demand-horizon\config\service_crosswalk.csv


## Rural urban classification

The Office for National Statistics Rural Urban Classification 2021 of fire and
rescue areas. It is an attribute table with no geometry covering England and
Wales, so it is fetched as a table rather than a layer and filtered to England.

The join uses FRA25 codes, so Hampshire and Isle of Wight carries E31000048 here
and E31000017 in the incident data, the same mismatch the crosswalk already
handles. The join is therefore on `boundary_code`, not `e_code_territory`. The
boundary layer carries FRA22 codes, so an authority whose code changed between
2022 and 2025 would show below as unclassified rather than be matched by name.

In [9]:
classification_source = config.sources["ons_fra_classification"][0]
table_path = raw_dir / "boundaries" / classification_source.filename
table_rows = bd.fetch_table(classification_source.url)
table_record = bd.save_table(table_rows, table_path)
table_record

{'filename': 'fra_2025_ruc21_ew.json',
 'features': 47,
 'properties': ['FRA25CD',
  'FRA25NM',
  'ObjectId',
  'Proportion_of_population_in_OAs_further_from_a_major_town_or_city____',
  'Proportion_of_population_in_rural_OAs____',
  'RUC21CD',
  'RUC21NM',
  'RUC21_relative_access',
  'RUC21_settlement_class',
  'Rural_Urban_flag'],
 'sha256': 'f449616724b0a0b300fcb695621a9989d81f7dc73b4827c48cccc8b99186474b',
 'bytes': 20026}

In [10]:
classification = bd.load_classification(table_path)
england = {code: row for code, row in classification.items() if code.startswith("E")}
print(f"{len(classification)} classified areas, {len(england)} in England")

47 classified areas, 44 in England


In [11]:
ruc = pd.DataFrame(list(england.values()))
classified = crosswalk.merge(
    ruc, how="left", left_on="boundary_code", right_on=contract.CLASSIFICATION_KEY
).sort_values(contract.RURAL_SHARE, ascending=False)
classified[
    [
        "e_code_territory",
        "frs_territory",
        "boundary_code",
        contract.SETTLEMENT_CLASS,
        contract.RURAL_SHARE,
    ]
]

,e_code_territory,frs_territory,boundary_code,RUC21_settlement_class,Proportion_of_population_in_rural_OAs____
7,E31000008,Cornwall and Isles of Scilly,E31000008,Majority rural,55.4
8,E31000009,Cumbria,E31000009,Majority rural,50.4
27,E31000026,Norfolk,E31000026,Intermediate rural,48.7
25,E31000025,Lincolnshire,E31000025,Intermediate rural,47.5
30,E31000029,Northumberland,E31000029,Intermediate rural,45.1
28,E31000027,North Yorkshire,E31000027,Intermediate rural,42.4
36,E31000034,Suffolk,E31000034,Intermediate rural,40.1
33,E31000032,Shropshire,E31000032,Intermediate rural,38.6
10,E31000011,Devon and Somerset,E31000011,Intermediate rural,37.9
12,E31000013,Durham,E31000013,Intermediate rural,37.9


In [12]:
unclassified = classified.loc[
    classified[contract.CLASSIFICATION_KEY].isna(),
    ["e_code_territory", "frs_territory", "boundary_code", "match_basis"],
]
print("crosswalk rows with no classification:")
print(unclassified.to_string(index=False) if len(unclassified) else "  none")

crosswalk rows with no classification:
  none
